In [1]:
import requests
import pandas as pd
from datetime import datetime

def get_secret(name, vault_env="OUTBUILD_KEYVAULT_URL"):
    """Key Vault in Fabric, environment variable locally.

    Mirrors get_secret() in src/procore/procore_extract.py - credentials live in
    exactly one place and never in the notebook source.
    """
    import os
    vault = os.environ.get(vault_env)
    if vault:
        return notebookutils.credentials.getSecret(vault, name)
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Secret {name!r} not found. Set {vault_env} to your Key Vault URL, "
            f"or export {name} locally. See foundation/README.md."
        )
    return value

TOKEN = get_secret("OUTBUILD_API_TOKEN")

BASE_URL = "https://datahub.outbuild.com"
HEADERS = {
    "authorizationToken": TOKEN,
    "Content-Type": "application/json"
}

def fetch_all_pages(endpoint: str, data_key: str) -> list:
    results = []
    page = 1
    while True:
        response = requests.get(f"{BASE_URL}/{endpoint}?page={page}", headers=HEADERS)
        response.raise_for_status()
        raw = response.json()
        body = raw.get("body", raw)
        records = body.get(data_key, [])
        results.extend(records)
        if not body.get("hasNextPage", False):
            break
        page += 1
    return results

# ----------------------------------------------------------
# Fetch all projects
# ----------------------------------------------------------
print("Fetching projects...")
projects_raw = fetch_all_pages("projects", "projects")
print(f"  {len(projects_raw)} projects found\n")

# ----------------------------------------------------------
# Fetch activities for every project
# ----------------------------------------------------------
all_activities = []

for project in projects_raw:
    project_id      = project["id"]
    project_name    = project["name"]
    procore_id      = project.get("procore_id", None)  # ← grab procore_id, null if not present

    print(f"  Fetching: {project_name} (outbuild_id: {project_id} | procore_id: {procore_id})")

    activities = fetch_all_pages(f"activities/project/{project_id}", "activities")
    print(f"    → {len(activities)} activities")

    for a in activities:
        a["outbuild_project_id"] = project_id
        a["project_name"]        = project_name
        a["procore_project_id"]  = procore_id  # ← stored on every activity row

    all_activities.extend(activities)

print(f"\nTotal activities: {len(all_activities)}")

StatementMeta(, 102e5465-75eb-4828-9450-9587167ead4c, 3, Finished, Available, Finished, False)

Fetching projects...
  9 projects found

  Fetching: City Harvest (outbuild_id: 43250 | procore_id: None)
    → 63 activities
  Fetching: E25-034 CITY HARVEST 150 52ND ST (outbuild_id: 43183 | procore_id: None)
    → 4 activities
  Fetching: Embankment Phase III (outbuild_id: 42540 | procore_id: None)
    → 137 activities
  Fetching: Sauna Lounge (outbuild_id: 39780 | procore_id: None)
    → 271 activities
  Fetching: Training Project (outbuild_id: 37687 | procore_id: 562949953807489)
    → 10 activities
  Fetching: PCNA - 711 11th Avenue (outbuild_id: 34755 | procore_id: None)
    → 49 activities
  Fetching: 360Lex (outbuild_id: 32828 | procore_id: 562949955225798)
    → 598 activities
  Fetching: Affect Test (outbuild_id: 32572 | procore_id: None)
    → 6 activities
  Fetching: Sandbox (outbuild_id: 32321 | procore_id: None)
    → 82 activities

Total activities: 1220


In [2]:
# ----------------------------------------------------------
# Save raw to Bronze lakehouse
# ----------------------------------------------------------
df_bronze = pd.DataFrame(all_activities)

# Ensure baseline columns exist and are typed as string
# (avoids VOID type issue in Fabric when all values are null)
for col in ["baseline_start_date", "baseline_end_date"]:
    if col not in df_bronze.columns:
        df_bronze[col] = None
    df_bronze[col] = df_bronze[col].astype(str).replace("None", None)

if "baseline_duration" not in df_bronze.columns:
    df_bronze["baseline_duration"] = None
df_bronze["baseline_duration"] = df_bronze["baseline_duration"].astype("float64")

df_bronze["ingested_at"] = datetime.utcnow().isoformat()

spark.createDataFrame(df_bronze) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_outbuild_activities")

print(f"✅ Saved {len(df_bronze)} rows to bronze_outbuild_activities")

StatementMeta(, 102e5465-75eb-4828-9450-9587167ead4c, 4, Finished, Available, Finished, False)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 1d19e538-32ca-4f72-9791-accc0a5d9ddc).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- id: long (nullable = true)
-- parent_id: string (nullable = true)
-- unique_id: string (nullable = true)
-- description: string (nullable = true)
-- name: string (nullable = true)
-- duration: double (nullable = true)
-- cost_budgeted: long (nullable = true)
-- progress: double (nullable = true)
-- constraint_date: string (nullable = true)
-- start_date: string (nullable = true)
-- end_date: string (nullable = true)
-- constraint_type: string (nullable = true)
-- activity_type: string (nullable = true)
-- correlative_id: lo